# CONNECT TO running server as client

In [1]:
# server has to run first (buildandrun powershell)

In [1]:
# OLD CODE - keep in case
# this code lets you send messages to the server but can not receive from server to jupyter client 
  
# import socketio
# sio = socketio.Client()
# sio.disconnect()

# uid = 'jupyter-client'

# @sio.event(namespace='/main')
# def connect():
#     print("✅ Connected to /main")


# # Variables to store received data
# latest_data = None
# message_log = []

# # jupyter client listener for server events 
# @sio.on('ex', namespace='/main')
# def on_ex(data):
#     global latest_data, message_log
#     print(f"📩 Server returned on 'ex': {data}")
#     latest_data = data
#     message_log.append({'event': 'ex', 'data': data}

#     # Example: auto-react
#     if data.get('fn') == 'dropdown' and data.get('id') == 'projDD':
#         print("🧠 Project updated:", data.get('val'))
 
# # establish a connection to the server as another client 
# sio.connect('http://127.0.0.1:5000', namespaces=['/main'])
# sio.emit('join', {'usr': uid}, namespace='/main')
# sio.emit('init-project', namespace='/main')
# -----------------------------------------------------------------------------------------

import socketio
import time
import threading

# Setup Socket.IO client
sio = socketio.Client()
uid = 'jupyter-client'

latest_data = None
message_log = []

# 1️⃣ Define listeners BEFORE connect
@sio.on('ex', namespace='/main')
def on_ex(data):
    global latest_data, message_log
    print(f"📩 Server returned on 'ex': {data}")
    latest_data = data
    message_log.append({'event': 'ex', 'data': data})

@sio.event(namespace='/main')
def connect():
    print("✅ Connected to /main")
    sio.emit('join', {'usr': uid}, namespace='/main')

# 2️⃣ Connect
sio.connect('http://127.0.0.1:5000', namespaces=['/main'])

# 3️⃣ Start background thread AFTER connect
def wait_forever():
    while True:
        time.sleep(1)

thread = threading.Thread(target=wait_forever)
thread.daemon = True
thread.start()

print("📡 Listening for messages in the background...")


📡 Listening for messages in the background...
✅ Connected to /main


📩 Server returned on 'ex': {'usr': 'jupyter-client', 'val': {'name': 'Teapot', 'layouts': ['layout1-teapot', 'layout2-teapot', 'layout3-teapot'], 'layoutsRGB': ['layout1-teapot', 'layout2-teapot', 'layout3-teapot'], 'links': ['layout1-teapot', 'layout2-teapot', 'layout3-teapot'], 'linksRGB': ['layout1-teapot', 'layout2-teapot', 'layout3-teapot'], 'selections': [], 'scenes': ['layout1-teapot', 'layout2-teapot', 'layout3-teapot'], 'info': 'A toy graph for testing purposes. Number of nodes: 51361, Links: 102560.', 'linkcount': 102560, 'labelcount': 0, 'nodecount': 51361, 'labels': [51361, 0], 'annotationTypes': True}, 'fn': 'project'}
📩 Server returned on 'ex': {'usr': 'jupyter-client', 'fn': 'layout', 'id': 'layoutExists', 'val': False}
📩 Server returned on 'ex': {'usr': 'jupyter-client', 'id': 'projDD', 'fn': 'dropdown', 'parent': 'projDD', 'sel': 12, 'name': 'Teapot'}
📩 Server returned on 'ex': {'id': 'x', 'success': 'true', 'fn': 'projectLoaded', 'usr': 'OjNh9GSvgC'}
📩 Server returned

### sending from jupyter-client to others (e.g. webclient)

In [2]:
# basic example 
# sio.emit('ex', {
#     'usr': 'jupyter-client',
#     'msg': 'Node: 20', 'id': None, 'val': '20', 'fn': 'node'
# }, namespace='/main')


### change project 

In [3]:
# get all projects in backend 

import GlobalData as GD

# print as pretty table with index from  0 - x
allprojects = []
for i, proj in enumerate(GD.listProjects()):
    allprojects.append((i,proj))
allprojects

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'diffusion'),
 (3, 'GenExpression_01'),
 (4, 'GenExpression_02'),
 (5, 'imunet_250130-X0-inter'),
 (6, 'imunet_250130-X1-inter'),
 (7, 'Interactive_Project_T01'),
 (8, 'JSON_autocore'),
 (9, 'JSON_barbellgraph'),
 (10, 'JSON_Zachary'),
 (11, 'Sphere_Torus'),
 (12, 'Teapot'),
 (13, 'Test'),
 (14, 'TheMandelbulb_edges')]

In [4]:
# choose a project id from list above to change project 
sel_index = 12

sel_id = allprojects[sel_index][0]
sel_name = allprojects[sel_index][1]

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')

### receive messages from server or web/VR client

In [5]:
import time 
#time.sleep(2) 

# latest  received data 
print("📦 Latest data received:", latest_data)

# message log 
#print("📚 Message log:")
#for msg in message_log:
#    print(msg


📦 Latest data received: {'val': {'id': 12342, 'n': 12342, 'attrlist': {'annot1': ['alpha', 'lambda', 'beta'], 'annot2': ['gamma', 'lambda'], 'annot3': ['nu', 'theta']}}, 'fn': 'node', 'id': '12342', 'nch': 4}


In [6]:
# # # select a node via jupyter client
sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'node',
    'val': '20',
    'msg' : 'Node: ',
    'id': None,
}, namespace='/main')

In [8]:
print("📦 Latest data received:", latest_data)


📦 Latest data received: {'usr': 'nsqFnu6nNM', 'fn': 'updateTempTex', 'textures': [{'channel': 'layoutNodesLow', 'path': 'static/projects/Teapot/layoutsl/templ.bmp'}, {'channel': 'layoutNodesHi', 'path': 'static/projects/Teapot/layouts/temp.bmp'}]}


### change layout (realtime calculation)